### Test on Video

##### Imports

In [1]:
from pathlib import Path
from ultralytics import YOLO
import torch
from torchvision import transforms, models
from torch.utils.data import DataLoader
import torch.nn as nn

##### Paths & Parameters

In [2]:
BALANCED_DIR = Path("../data/processed/resnet18_dataset_balanced")
MODEL_DIR = Path('../models')

yolo_model = YOLO(MODEL_DIR / "best.pt")
state_dict = torch.load(MODEL_DIR / "fluxia_classifier.pth")
class_names = torch.load(MODEL_DIR / "class_names.pth")


IMG_SIZE = 224
NORMALIZE_MEAN = [0.485, 0.456, 0.406]
NORMALIZE_STD = [0.229, 0.224, 0.225]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


##### Recreate ResNet architecture

In [3]:
classifier = models.resnet18(pretrained=False)

num_features = classifier.fc.in_features
classifier.fc = nn.Linear(num_features, 4)   # 4 classes

classifier.load_state_dict(state_dict)
classifier.to(DEVICE)
classifier.eval()

c:\Users\anass\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\anass\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

##### Data Transforms

In [4]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        NORMALIZE_MEAN,
        NORMALIZE_STD
    )
])

##### Video Test

In [ ]:
import cv2
import torch
import numpy as np
from PIL import Image

def process_video(
    video_path,
    yolo_model,
    classifier,
    transform,
    class_names,
    device,
    margin=0.3
):
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print("Error opening video")
        return
    
    boxes = None
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        h, w = frame.shape[:2]

        # -------------------------
        # YOLO Detection
        # -------------------------
        results = yolo_model(frame, conf=0.5)

        if len(results[0].boxes) > 0:
            boxes = results[0].boxes

        if boxes is not None:
            
            for box in boxes:

                # YOLO outputs XYXY
                x1, y1, x2, y2 = map(int, box.xyxy[0])

                crop = frame[y1:y2, x1:x2]

                # -------------------------
                # Add Margin Safely
                # -------------------------
                box_w = x2 - x1
                box_h = y2 - y1

                x_pad = int(box_w * margin)
                y_pad = int(box_h * margin)

                x1_new = max(0, x1 - x_pad)
                y1_new = max(0, y1 - y_pad)
                x2_new = min(w, x2 + x_pad)
                y2_new = min(h, y2 + y_pad)

                crop = frame[y1_new:y2_new, x1_new:x2_new]

                if crop.size == 0:
                    continue

                # -------------------------
                # Classification
                # -------------------------
                pil_crop = Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))

                input_tensor = transform(pil_crop).unsqueeze(0).to(device)

                with torch.no_grad():

                    outputs = classifier(input_tensor)
                    _, predicted_class = torch.max(outputs, 1)

                label = class_names[predicted_class.item()]

                # -------------------------
                # Draw Results
                # -------------------------
                # cv2.rectangle(frame, (x1_new, y1_new), (x2_new, y2_new), (0, 255, 0), 2)
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

                cv2.putText(
                    frame,
                    label,
                    (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0, 255, 0),
                    2
                )
        # else:
            # -------------------------
            # YOLO Detection
            # -------------------------
            # results = yolo_model(frame, conf=0.5)

            # if len(results[0].boxes) > 0:
            #     boxes = results[0].boxes

        # -------------------------
        # Display Frame
        # -------------------------
        cv2.imshow("Fluxia Pipeline", frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()

In [23]:
process_video(
    video_path=r"C:\Users\anass\Desktop\kling_20260220_Text_to_Video_Static_ove_6107_0.mp4",
    yolo_model=yolo_model,
    classifier=classifier,
    transform=transform,
    class_names=class_names,
    device=DEVICE
)


0: 384x640 6 tables, 9.7ms
Speed: 2.9ms preprocess, 9.7ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 tables, 13.1ms
Speed: 2.2ms preprocess, 13.1ms inference, 2.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 tables, 7.7ms
Speed: 1.8ms preprocess, 7.7ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 tables, 9.8ms
Speed: 2.3ms preprocess, 9.8ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 tables, 13.2ms
Speed: 2.5ms preprocess, 13.2ms inference, 2.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 tables, 10.7ms
Speed: 2.3ms preprocess, 10.7ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 tables, 9.6ms
Speed: 1.8ms preprocess, 9.6ms inference, 2.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 tables, 8.6ms
Speed: 1.2ms preprocess, 8.6ms inference, 2.4ms postprocess per image at shape (1, 3, 384, 640)

0

##### Image Test

In [5]:
import cv2
import torch
from PIL import Image
import matplotlib.pyplot as plt

def test_images(
    image_path,
    yolo_model,
    classifier,
    transform,
    class_names,
    device,
    margin=0.3
):
    frame = cv2.imread(image_path)
    
    boxes = None

    h, w = frame.shape[:2]

    # -------------------------
    # YOLO Detection
    # -------------------------
    results = yolo_model(frame, conf=0.5)

    if len(results[0].boxes) > 0:
        boxes = results[0].boxes

    if boxes is not None:
        
        for box in boxes:

            # YOLO outputs XYXY
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            crop = frame[y1:y2, x1:x2]

            # -------------------------
            # Add Margin Safely
            # -------------------------
            box_w = x2 - x1
            box_h = y2 - y1

            x_pad = int(box_w * margin)
            y_pad = int(box_h * margin)

            x1_new = max(0, x1 - x_pad)
            y1_new = max(0, y1 - y_pad)
            x2_new = min(w, x2 + x_pad)
            y2_new = min(h, y2 + y_pad)

            crop = frame[y1_new:y2_new, x1_new:x2_new]

            if crop.size == 0:
                continue

            # -------------------------
            # Classification
            # -------------------------
            pil_crop = Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))

            input_tensor = transform(pil_crop).unsqueeze(0).to(device)

            with torch.no_grad():

                outputs = classifier(input_tensor)
                _, predicted_class = torch.max(outputs, 1)

            label = class_names[predicted_class.item()]

            # -------------------------
            # Draw Results
            # -------------------------
            # cv2.rectangle(frame, (x1_new, y1_new), (x2_new, y2_new), (0, 255, 0), 2)
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

            cv2.putText(
                frame,
                label,
                (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 0),
                2
            )
    # else:
        # -------------------------
        # YOLO Detection
        # -------------------------
        # results = yolo_model(frame, conf=0.5)

        # if len(results[0].boxes) > 0:
        #     boxes = results[0].boxes

    # -------------------------
    # Display Frame
    # -------------------------

    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    plt.imshow(frame)
    plt.axis("off")
    plt.show()

In [ ]:
import os

dir_name = r"C:\Users\anass\Desktop\dd\\"

images = os.listdir(dir_name)

for img in images:

    image_path = dir_name + img

    test_images(
        image_path, 
        yolo_model=yolo_model,
        classifier=classifier,
        transform=transform,
        class_names=class_names,
        device=DEVICE
    )